# 8. Disturbance Rejection

[Previous lesson](07_controller_tuning.ipynb) · [Next lesson](09_practical_pid.ipynb)

Tracking a new target and recovering from an external load are different experiments. Hold the target fixed while applying a known torque disturbance and compare the recovery.

Work through the explanations, then implement each exercise. Hints and complete reference solutions are collapsed below your code cells. Solutions build in order; Run All does not execute them.

**Prerequisites:** complete the preceding lessons. This notebook starts independently: run the setup cell once before attempting the exercises.

## Previously implemented code

This setup repeats earlier material so you do not need another notebook running. Read it as a recap, then run it. The new work for this lesson remains in the empty cells below.

**Setup inventory:** `motor_dynamics` evaluates the two motor derivatives using SI units and state order `[current, omega]`. Where supplied, `simulate` advances the sampled controller and motor separately, and `plot_run` displays speed, voltage, and term contributions. Controller classes in setup were taught previously; their histories start fresh when you construct an instance. Run setup after restarting the kernel, then execute your exercise cells from top to bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

R, L, K_t, K_e, J, b = 1.0, 0.01, 0.1, 0.1, 0.01, 0.001

def motor_dynamics(t, state, voltage, load_torque):
    current, omega = state
    return [(voltage - R*current - K_e*omega)/L,
            (K_t*current - b*omega - load_torque)/J]
class PID:
    def __init__(self, kp, ki, kd):
        self.kp, self.ki, self.kd = kp, ki, kd
        self.reset()
    def reset(self):
        self.integral = 0.0
        self.previous_error = None
        self.p_term = self.i_term = self.d_term = 0.0
    def update(self, setpoint, measurement, dt):
        if not np.isfinite(dt) or dt <= 0:
            raise ValueError("dt must be finite and positive")
        error = setpoint-measurement
        slope = 0.0 if self.previous_error is None else (error-self.previous_error)/dt
        self.integral += error*dt
        self.previous_error = error
        self.p_term = self.kp*error
        self.i_term = self.ki*self.integral
        self.d_term = self.kd*slope
        return self.p_term+self.i_term+self.d_term
def simulate(controller, target=lambda t: 40.0, load=lambda t: 0.0,
             duration=8.0, dt=0.001, noise=0.0, seed=42):
    # Each row describes the beginning of one control interval.
    if not np.isfinite(dt) or dt <= 0:
        raise ValueError("dt must be finite and positive")
    if not np.isfinite(duration) or duration <= 0:
        raise ValueError("duration must be finite and positive")
    if not np.isfinite(noise) or noise < 0:
        raise ValueError("noise must be finite and nonnegative")
    ratio = duration/dt
    if not np.isfinite(ratio):
        raise ValueError("duration/dt must be finite")
    count = round(ratio)
    if count < 1 or not np.isclose(ratio, count, rtol=0, atol=1e-9):
        raise ValueError("duration must be a positive integer multiple of dt")
    rng = np.random.default_rng(seed)
    state = np.array([0.0, 0.0])
    records = []
    for k in range(count):
        t = k*dt
        reference = target(t)
        measured = state[1] + noise*rng.normal()
        voltage = controller.update(reference, measured, dt)
        requested = getattr(controller, "requested", voltage)
        records.append([t, reference, state[0], state[1], measured,
                        requested, voltage, controller.p_term,
                        controller.i_term, controller.d_term])
        result = solve_ivp(motor_dynamics, (t, t+dt), state,
                          args=(voltage, load(t)), rtol=1e-7, atol=1e-9)
        if not result.success:
            raise RuntimeError(result.message)
        state = result.y[:, -1]
    keys = "time target current speed measured requested voltage p i d".split()
    return dict(zip(keys, np.asarray(records).T))

def plot_run(run, title):
    fig, axes = plt.subplots(3, 1, sharex=True, figsize=(10, 8), layout="constrained")
    axes[0].plot(run['time'], run['target'], '--', label='Target')
    axes[0].plot(run['time'], run['speed'], label='True speed')
    axes[0].set_ylabel('Speed (rad/s)')
    axes[1].plot(run['time'], run['requested'], label='Requested')
    axes[1].plot(run['time'], run['voltage'], '--', label='Applied')
    axes[1].set_ylabel('Voltage (V)')
    for key in ['p', 'i', 'd']:
        axes[2].plot(run['time'], run[key], label=key.upper())
    axes[2].set_ylabel('Contribution (V)')
    axes[2].set_xlabel('Time (s)')
    for ax in axes:
        ax.grid(alpha=.25); ax.legend()
    fig.suptitle(title)
    plt.show()


<img src="assets/lesson-08.svg" alt="Load disturbance enters the torque balance" style="display:block; width:100%; height:auto; margin:0 auto;">

<div class="figure-caption" style="background-color:rgba(128,128,128,0.10);padding:12px 18px;">

**Load disturbance.** Added load reduces net rotor torque before feedback can react. The target remains fixed; the controller must supply additional effort to recover.

</div>

## 8.1 A disturbance enters the plant

Load torque enters the mechanical equation directly:

$$J\dot\omega=K_t i-b\omega-\tau_L.$$

An increase in load initially makes net torque negative. Speed begins falling; feedback detects the change and raises voltage. At 40 rad/s, an additional 0.1 N·m load requires an extra 1 A of steady current and an extra 1 V of resistive drop.

### Follow the physical sequence

Before the disturbance, motor torque approximately balances friction. At the load step, current and speed remain continuous because inductance and inertia prevent instantaneous jumps under finite inputs. The new load changes acceleration immediately by $-\Delta\tau_L/J$.

For a 0.1 N·m step and $J=0.01$, that change is −10 rad/s². The controller only responds after a sampled measurement reflects the falling speed. Higher voltage then changes current, current changes torque, and speed recovers over time.

At the same desired speed, back EMF is unchanged. The extra steady voltage is needed for extra resistive drop due to the added current. Required current increases by $\Delta\tau_L/K_t=1$ A, so voltage increases by $R\Delta i=1$ V. This explains the 4.4→5.4 V equilibrium shift without needing to inspect a plot first.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 1 — Calculate loaded effort

Calculate the steady current and voltage needed at 40 rad/s with loads 0 and 0.1 N·m.

</div>

<details>
<summary>Show a hint</summary>

First use torque balance for current, then the voltage balance.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
for load in [0.0,.1]:
    current_ss=(b*40+load)/K_t
    voltage_ss=R*current_ss+K_e*40
    print('Load:',load,'current:',current_ss,'voltage:',voltage_ss)
np.testing.assert_allclose(R*.1/K_t,1.0)
```

The loaded target needs 5.4 V instead of 4.4 V, before considering transient demand.

</details>

## 8.2 Keep the comparison controlled

Apply the load at eight seconds so the preceding startup has time to settle. Use the same target, motor, time interval, and disturbance for each controller. Compare P, PI, and PID; remember that P already has a baseline tracking error before the disturbance.

### Establish the baseline before measuring the disturbance

A disturbance during startup would mix two phenomena: initial acceleration and recovery from a new load. Apply the event after eight seconds and inspect the preceding interval to confirm the response has nearly settled.

Pure P control has a baseline below 40 rad/s even before the load. PI and PID have baselines near 40. Compare both the full curves and a close-up around eight seconds; a large startup range can hide a small disturbance dip.

The same load schedule must be used for every controller. Make the event exactly coincide with a sample boundary so the held load used during each integration interval is unambiguous.

Keep derivative gain zero for the PI run rather than using a different motor or solver. Only the controller configuration should distinguish the cases.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 2 — Compare disturbance responses

Run (Kp,Ki,Kd)=(0.4,0,0), (0.4,0.5,0), and (0.4,0.5,0.01), applying 0.1 N·m at t=8 and ending at t=16. Overlay speed curves. Also plot speed minus its pre-disturbance baseline over 7.5–11 seconds to compare the disturbance dips directly.

</div>

<details>
<summary>Show a hint</summary>

Keep a separate result per controller and mark the disturbance time.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
disturbed={}
for name,gains in [('P',(.4,0,0)),('PI',(.4,.5,0)),('PID',(.4,.5,.01))]:
    trial=simulate(PID(*gains),duration=16,load=lambda t:0.0 if t<8 else .1)
    disturbed[name]=trial
    plt.plot(trial['time'],trial['speed'],label=name)
plt.axvline(8,ls=':',color='gray');plt.axhline(40,ls='--',color='gray')
plt.xlabel('Time (s)');plt.ylabel('Speed (rad/s)');plt.legend();plt.show()
fig,ax=plt.subplots(figsize=(10,4),layout='constrained')
for name,trial in disturbed.items():
    baseline=trial['speed'][(trial['time']>=7.5)&(trial['time']<8)].mean()
    window=(trial['time']>=7.5)&(trial['time']<=11)
    ax.plot(trial['time'][window],trial['speed'][window]-baseline,label=name)
ax.set(xlim=(7.5,11),xlabel='Time (s)',ylabel='Speed change from baseline (rad/s)')
ax.axvline(8,ls=':',color='gray');ax.legend();ax.grid(alpha=.25);plt.show()

```

A P controller can recover to a new equilibrium while still missing the target. Do not confuse recovery with zero tracking error.

</details>

## 8.3 Measure drop, recovery, and residual

Measure disturbance drop relative to the pre-disturbance mean speed, not automatically relative to the target. Define recovery as settling back inside ±2% of that baseline for the remainder of the observation. Separately report final target error.

For PI/PID, the baseline should be close to the target. For P, its distinct baseline makes the distinction essential.

### Two reference levels answer different questions

Suppose a controller is at 31.4 rad/s before a disturbance, then dips to 29.4. Its disturbance drop is 2 rad/s relative to baseline, while its instantaneous target error is 10.6 rad/s relative to 40. Reporting only one number would conceal the other behavior.

Our recovery band is ±2% of the baseline magnitude. For 40 rad/s that is ±0.8; for 31.4 it is about ±0.628. State the reference explicitly whenever comparing recovery times.

Use the last sample outside the band after the disturbance, then take the next sample's time minus the event time. If the run ends outside the band, recovery was not observed. This avoids calling the first brief crossing a completed recovery.

A P controller may settle to a new lower equilibrium and never re-enter its old baseline band. That is a valid measured outcome, not an exception to suppress.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 3 — Measure recovery

Implement disturbance_metrics. Use t=7.5–8 for baseline, t≥8 for the event, and the final 0.5 s for target error. Report unavailable recovery when the last sample is outside the band.

</div>

<details>
<summary>Show a hint</summary>

Reuse the “last outside sample” pattern, but subtract the event time.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
def disturbance_metrics(run):
    t,y=run['time'],run['speed']
    baseline=y[(t>=7.5)&(t<8)].mean()
    ids=np.flatnonzero(t>=8)
    outside=ids[np.abs(y[ids]-baseline)>.02*abs(baseline)]
    recovery=0.0 if len(outside)==0 else None if outside[-1]==len(t)-1 else t[outside[-1]+1]-8
    return dict(drop=float(baseline-y[ids].min()),recovery=recovery,
                final_error=float(40-y[t>=15.5].mean()))
for name,trial in disturbed.items():print(name,disturbance_metrics(trial))
```

This definition explicitly separates recovery to baseline from tracking accuracy relative to the requested target.

</details>

## 8.4 Explain recovery through voltage

Integral action accumulates the negative effect of the load as positive speed error and raises its stored voltage contribution. Once speed recovers, the integral retains the additional command needed for the load.

Derivative action reacts to the changing measurement; it cannot by itself supply a persistent correction at constant error.

<div class="lesson-callout" style="border-left: 3px solid #b99a62; background-color: rgba(185, 154, 98, 0.08); padding: 12px 18px; margin: 16px 0;">

**Key distinction**

Disturbance rejection is limited by available actuator effort. The present ideal model can request any voltage; the next lesson will impose a real bound.

</div>

### Explain recovery through the controller terms

Immediately after the speed begins falling, proportional error increases and raises the command. Integral action grows as the positive error persists. Once the speed recovers, P shrinks again, but I retains the additional voltage needed for the new load.

A derivative term reacts to the changing error during the dip and recovery. At a constant equilibrium its contribution tends to zero. It cannot retain the new steady command by itself.

In the supplied comparison, PID slightly reduces the maximum drop but takes slightly longer than PI to satisfy the chosen recovery criterion. That is a useful tradeoff rather than a reason to declare one controller universally superior.

Finally compare the loaded voltage with the future 6 V limit. The 5.4 V steady requirement fits, but transient requested voltages can still exceed the limit. Feasible equilibrium does not mean the unlimited-actuator transient remains achievable.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 4 — Validate loaded equilibrium

Plot the PI run’s terms and voltage. Verify PI and PID final errors are below 0.1 rad/s. Compare P final speed with the loaded analytical P equilibrium.

</div>

<details>
<summary>Show a hint</summary>

Use the P error formula with load=0.1 and the same target.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
plot_run(disturbed['PI'],'PI recovery from external load')
for name in ['PI','PID']:
    assert abs(disturbance_metrics(disturbed[name])['final_error'])<.1
expected_error=((R*b+K_t*K_e)*40+R*.1)/(R*b+K_t*K_e+K_t*.4)
np.testing.assert_allclose(40-disturbed['P']['speed'][-1],expected_error,rtol=.01)
```

The final loaded integral voltage approaches 5.4 V. This is the new effort required for the same speed, not a change in the target.

</details>

## What the experiment establishes

The P controller settles below target after the load step. PI and PID recover near the target by increasing their persistent voltage contribution. For these gains, PID slightly reduces the initial drop but does not improve the recovery-time metric. Next we will impose voltage limits and protect the internal state.

## Before moving on

Run your exercises from a fresh kernel in order. Check the stated numerical conditions and explain your plots in your own words: what changed, why it changed, and which model assumptions limit the conclusion. Reference solutions are one implementation, not the only correct approach.

[Previous lesson](07_controller_tuning.ipynb) · [Next lesson](09_practical_pid.ipynb)

## References and further reading

[Åström and Murray, Feedback Systems, Chapter 11](https://www.cds.caltech.edu/~murray/books/AM08/pdf/fbs-pid_24Jul2020.pdf) — additional background on PID structure and implementation. Our examples use independent parallel gains and the conventions stated in this lesson.